# quant-garage demo session

One notebook, five workflows. What a real research session looks like.

You provide:
- A `MASSIVE_API_KEY` environment variable (free Basic tier is enough for most of this).
- Your positions or a watchlist (we'll use a stand-in throughout).

The framework provides everything else. Every cell renders sell-side-quality output cited to a live API call.

**Suggested reading order**
1. Setup (below)
2. Morning brief (60 seconds, macro + news)
3. Weekly brief (Sunday-night, calendar + rotation)
4. Portfolio review (full 8-tool workflow on a book)
5. Preflight trade (before you hit execute on a specific name)
6. Historical comparison (twin decision-support)

Skip around freely. Every cell is independent.

## Setup

Install the package (once):

```bash
pip install quant-garage
```

Set your Massive key:

```bash
export MASSIVE_API_KEY=...
```

Then in Python:

In [ ]:
# One import surface for everything.
from quant_garage.skills import (
    morning_brief,
    weekly_brief,
    portfolio_review,
    preflight_trade,
    historical_comparison,
    fixed_income_context,
)

# Every skill exposes the same two entry points:
#   run(...)     -> dict         (canonical JSON payload for downstream code)
#   render(dict) -> str          (human-facing briefing)
print('quant-garage loaded.')

## 1. Morning brief

60-second daily open. Chains:
- `market-regime` (SPY trend, VIX, breadth, sector leadership)
- `macro-event-calendar` (today + tomorrow, historical |SPY move| per event type)
- `news-scanner` (last N per watchlist ticker)

Cron this at 8am and read it while your coffee brews. Watchlist is optional — skip it for macro-only.

In [ ]:
payload = morning_brief.run(watchlist='NVDA,ALLO,SOFI', news_last_n=3)
print(morning_brief.render(payload))

**What just happened**: the skill pulled SPY, the 11 SPDR sector ETFs (for breadth + leadership), the macro release calendar, and up to 3 recent news articles per watchlist ticker. Cross-referenced sentiment vs price reaction. Emitted a headline block on top and full per-section detail below.

The `payload` dict is fully structured — wire it straight into a dashboard or agent.

In [ ]:
# Peek at the headline JSON directly
payload['headline']

## 2. Weekly brief

Sunday-night prep for the week ahead. Chains:
- `market-regime`
- `sector-rotation-signal` (30d rank change across the 11 SPDR sector ETFs)
- `macro-event-calendar` (7d forward)
- `earnings-blackout` (7d forward, watchlist)

Different from morning-brief: this one's about the WEEK's setup, not the day's news.

In [ ]:
watchlist = 'NVDA,ALLO,SOFI,HOOD,QCOM,BRK.B,GLD'

payload = weekly_brief.run(watchlist=watchlist, window_days=7)
print(weekly_brief.render(payload))

## 3. Portfolio review

The full book review in one command. Chains 8 sub-skills into a single briefing.

You provide positions as `TICKER=WEIGHT` (weights should sum to ~1.0) plus book value in dollars. The rebalancer uses the book value to size the trade tickets it recommends.

This is the workflow that was built after a live review missed an ALLO public offering (34% dilution) because it was run manually and one tool was skipped. Now the workflow is one command that doesn't skip.

In [ ]:
# Stand-in book. Replace with your real weights.
positions = 'JEPI=0.30,BRK.B=0.20,GLD=0.15,VTI=0.15,NVDA=0.08,SOFI=0.07,ALLO=0.05'
book_value = 500_000

payload = portfolio_review.run(
    positions=positions,
    book_value=book_value,
    # Rebalancer caps (feel free to tune)
    max_variance_share=0.25,
    max_weight=0.15,
    max_churn=0.10,
)
print(portfolio_review.render(payload))

**What's in the headline**
- Regime label (from market-regime)
- Rotation theme (from sector-rotation-signal)
- 90-day forward SPY distribution (from historical-analog-finder)
- Portfolio vol + top variance contributor (from risk-report)
- Next 3 earnings prints (from earnings-blackout)
- Next high-impact macro event (from macro-event-calendar)
- Top corporate action by |T+5 abnormal| (from corporate-actions-scanner)
- Rebalance verdict + biggest trim (from portfolio-rebalancer)

Below the headline: each section rendered in full by its own skill's render() helper.

In [ ]:
# The rebalancer's trade tickets in raw JSON.
# In practice: pipe these through your broker's API for a dry run.
rb = payload['sections']['rebalancer']
if rb:
    for t in rb['trades'][:5]:
        print(f"{t['ticker']:<8} {t['action'].upper():<5} ${abs(t['delta_dollar']):>10,.0f}  "
              f"weight {t['weight_before']*100:.1f}% -> {t['weight_after']*100:.1f}%")

## 4. Preflight trade

Before you hit execute on a specific name. You provide ticker + intended action (buy, sell, add, reduce, exit). Chains:
- `technical-briefing`
- `earnings-blackout` (14d forward)
- `news-scanner` (last N)
- `corporate-actions-scanner` (90d back)

Returns a deterministic verdict (`go`, `wait`, `review`) plus red and green flag lists. Not a recommendation to trade or not trade. A structured "is now obviously a bad time" gate.

In [ ]:
payload = preflight_trade.run(ticker='ALLO', action='reduce')
print(preflight_trade.render(payload))

In [ ]:
# The verdict + flags in raw JSON.
print('Verdict:', payload['verdict'])
print('Reds:', payload['red_flags'])
print('Greens:', payload['green_flags'])

## 5. Historical comparison

Twin decision-support. `event-study` on a specific event + `historical-analog-finder` on the market regime. Both anchors together so you're not relying on one.

Analog-only mode (skip the event side) if you don't have a specific event to study.

In [ ]:
# Event mode: NVDA's most recent earnings
payload = historical_comparison.run(
    ticker='NVDA',
    event_class='earnings',
    period='most_recent',
    analog_k=20,
)
print(historical_comparison.render(payload))

## 6. Fixed-income context

Rates and credit view via ETF proxies (SHV, SHY, IEF, TLT, TIP, LQD, HYG, AGG). Every equity valuation implicitly assumes something about rates; this skill closes that gap without needing FRED.

Returns:
- Per-proxy returns across 1/5/20/60/120 day windows
- Percentile of current price within trailing year
- HYG-LQD credit spread delta (positive = credit tightening)
- TLT-IEF duration spread delta
- HYG-SPY 60d rolling correlation
- Regime label (risk_off, credit_stress, goldilocks, reflation, rate_pressure, neutral)

In [ ]:
payload = fixed_income_context.run()
print(fixed_income_context.render(payload))

## Composing your own workflow

Every skill is a standalone `run(...) -> dict`. Chain them yourself:

In [ ]:
from quant_garage.skills import market_regime, sector_rotation_signal, fixed_income_context

# A three-tool macro brief in five lines.
regime = market_regime.run()
rotation = sector_rotation_signal.run(rotation_window=30)
rates = fixed_income_context.run()

print('Equity regime:', regime['composite_regime']['label'])
print('Rotation theme:', rotation['theme_read'])
print('Rates regime:', rates['regime']['label'])
print('Read:', rates['regime']['read'])

## Where to go from here

- **Full tool list**: [README.md](../README.md) in the repo has all 26 primitives + 8 workflows with per-tool descriptions.
- **Every skill has three surfaces**: Python (this notebook), CLI (`examples/run-*.py`), and Claude Code discovery (`skills/*/`).
- **JSON schemas**: every `run()` payload matches a JSON Schema at `skills/<name>/output-schema.json`. Wire the schemas into any tool-use LLM to get typed outputs for free.
- **Contributing**: [CONTRIBUTING.md](../CONTRIBUTING.md) for adding a skill. The audit gate (`npm run audit:requires`) enforces the dual-layer contract.